# Assignment 3: Retail Data Integration and Analysis
**Course**: R Programming (Sem 7, BE Computer Engineering)  
**Author**: Purva Gaonkar  
**Task**: Single lab: multi-source retail sales data integration and analysis.  

This notebook contains the complete pipeline for downloading the UCI Online Retail dataset, splitting it into three raw sources (CSV, JSON, and Excel), importing and cleaning the data, integrating the sources via dplyr, conducting sales and customer segmentation analysis, and finally storing the data in a SQLite database for SQL-based verification.

## Setup: Package Installation
We check and install the required R packages (`readr`, `readxl`, `writexl`, `jsonlite`, `dplyr`, `ggplot2`, `DBI`, `RSQLite`) if they are not already installed. This ensures the notebook runs seamlessly in any environment (including Google Colab).

In [ ]:
# Install packages if not present
required_packages <- c("readr", "readxl", "writexl", "jsonlite", "dplyr", "ggplot2", "DBI", "RSQLite")
new_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if (length(new_packages) > 0) {
  install.packages(new_packages, repos = "https://cloud.r-project.org")
}

# Load libraries
library(readr)
library(readxl)
library(writexl)
library(jsonlite)
library(dplyr)
library(ggplot2)
library(DBI)
library(RSQLite)

## Step 0: Data Source Preparation (`prepare_sources.R`)
We download the raw UCI Online Retail Excel dataset (roughly 23MB, 541,909 rows, 8 columns) and save it to `data/`. We verify the raw row count and split the sheet into three normalized sources:
1. `data/transactions.csv` (all rows; columns: InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate)
2. `data/products.json` (one row per StockCode; columns: StockCode, Description, UnitPrice using median price and modal description)
3. `data/customers.xlsx` (one row per non-NA CustomerID; columns: CustomerID, Country using modal country)

We print the number of keys with multiple distinct prices, descriptions, and countries, and report the generated dimensions.

In [ ]:
# Create data directory
if (!dir.exists("data")) {
  dir.create("data")
}

raw_path <- "data/Online Retail.xlsx"
url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

# Download the file if it is absent
if (!file.exists(raw_path)) {
  cat("Downloading Online Retail dataset from UCI...\n")
  tryCatch({
    download.file(url, raw_path, mode = "wb")
  }, error = function(e) {
    cat("Primary URL failed. Trying backup URL...\n")
    alt_url <- "https://web.archive.org/web/20231128032742/https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
    download.file(alt_url, raw_path, mode = "wb")
  })
} else {
  cat("Raw dataset already exists at:", raw_path, "\n")
}

# Load the dataset
cat("Loading Excel file...\n")
df <- read_excel(raw_path)

# Verify row count
expected_rows <- 541909
actual_rows <- nrow(df)
cat("Raw row count:", actual_rows, "\n")
if (actual_rows != expected_rows) {
  stop(paste("Row count verification failed! Expected:", expected_rows, "but got:", actual_rows))
} else {
  cat("Row count verified successfully.\n")
}

# Helper function to find the mode (most frequent value)
get_mode <- function(x) {
  x <- x[!is.na(x) & x != ""]
  if (length(x) == 0) return(NA_character_)
  tbl <- table(x)
  modes <- names(tbl)[tbl == max(tbl)]
  sort(modes)[1] # deterministic tie-breaker
}

# 1. Create transactions.csv
transactions <- df %>% select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate)
write.csv(transactions, "data/transactions.csv", row.names = FALSE)

# 2. Create products.json (collapsed by StockCode)
product_diagnostics <- df %>% 
  group_by(StockCode) %>% 
  summarise(
    distinct_prices = n_distinct(UnitPrice, na.rm = TRUE),
    distinct_descriptions = n_distinct(Description, na.rm = TRUE)
  )

cat("Number of StockCodes with >1 distinct price:", sum(product_diagnostics$distinct_prices > 1), "\n")
cat("Number of StockCodes with >1 distinct description:", sum(product_diagnostics$distinct_descriptions > 1), "\n")

products <- df %>% 
  group_by(StockCode) %>% 
  summarise(
    Description = get_mode(Description),
    UnitPrice = median(UnitPrice, na.rm = TRUE),
    .groups = "drop"
  )
jsonlite::write_json(products, "data/products.json", pretty = TRUE)

# 3. Create customers.xlsx (collapsed by CustomerID, exclude NA CustomerID)
customer_diagnostics <- df %>% 
  filter(!is.na(CustomerID)) %>% 
  group_by(CustomerID) %>% 
  summarise(distinct_countries = n_distinct(Country, na.rm = TRUE))

cat("Number of CustomerIDs with >1 distinct country:", sum(customer_diagnostics$distinct_countries > 1), "\n")

customers <- df %>% 
  filter(!is.na(CustomerID)) %>% 
  group_by(CustomerID) %>% 
  summarise(Country = get_mode(Country), .groups = "drop")
writexl::write_xlsx(customers, "data/customers.xlsx")

cat("Generated file dimensions:\n")
cat("- transactions.csv :", nrow(transactions), "x", ncol(transactions), "\n")
cat("- products.json    :", nrow(products), "x", ncol(products), "\n")
cat("- customers.xlsx   :", nrow(customers), "x", ncol(customers), "\n")

## Task 1: Import and Clean
We load the three split data formats (`readr::read_csv` for CSV, `jsonlite::fromJSON` for JSON, and `readxl::read_excel` for Excel). We perform data cleaning by:
- Removing duplicate records.
- Filtering out invalid or zero quantities (keeping positive quantities or negative quantities only if they represent cancellations - starting with "C").
- Filtering out products with invalid or zero unit prices (prices <= 0).

*Note on Missing CustomerIDs:* CustomerID is absent in approximately 25% of transactions. We choose to retain these transactions to ensure accurate revenue calculations. Dropping them would lead to underestimating total sales revenue.

*Note on Revenue Column:* Since the unit price of products is stored in `products.json` and not in `transactions.csv`, the `Revenue` column cannot be computed at this stage. It will be computed immediately after the integration in Task 2. This is noted explicitly to explain why the assignment's proposed order cannot be followed literally.

In [ ]:
# Import sources
transactions_raw <- read_csv("data/transactions.csv", show_col_types = FALSE)
products_raw <- jsonlite::fromJSON("data/products.json")
customers_raw <- read_excel("data/customers.xlsx")

# Create cleaning log dataframe
cleaning_log <- data.frame(
  Table = character(),
  Step = character(),
  Before_Rows = integer(),
  After_Rows = integer(),
  Removed = integer(),
  stringsAsFactors = FALSE
)

log_step <- function(table_name, step_desc, before_df, after_df) {
  bef <- nrow(before_df)
  aft <- nrow(after_df)
  rem <- bef - aft
  cleaning_log <<- rbind(cleaning_log, data.frame(
    Table = table_name,
    Step = step_desc,
    Before_Rows = bef,
    After_Rows = aft,
    Removed = rem,
    stringsAsFactors = FALSE
  ))
}

# Clean transactions
transactions_step1 <- distinct(transactions_raw)
log_step("Transactions", "Remove Duplicates", transactions_raw, transactions_step1)

transactions_step2 <- transactions_step1 %>% 
  filter(Quantity > 0 | (Quantity < 0 & grepl("^C", InvoiceNo)))
log_step("Transactions", "Filter Invalid Quantities", transactions_step1, transactions_step2)

transactions_cleaned <- transactions_step2
log_step("Transactions", "Retain NA CustomerID", transactions_step2, transactions_cleaned)

# Clean products
products_step1 <- distinct(products_raw)
log_step("Products", "Remove Duplicates", products_raw, products_step1)

products_cleaned <- products_step1 %>% filter(UnitPrice > 0)
log_step("Products", "Filter Zero/Negative Prices", products_step1, products_cleaned)

# Clean customers
customers_step1 <- distinct(customers_raw)
log_step("Customers", "Remove Duplicates", customers_raw, customers_step1)

customers_cleaned <- customers_step1
log_step("Customers", "No-op (Already clean)", customers_step1, customers_cleaned)

print(cleaning_log)

## Task 2: Integrate
We join the three sources using `dplyr::left_join`. We choose a left join with transactions as the primary table to retain all sales transactions. An inner join would drop all transactions with missing CustomerIDs (roughly 25% of the rows) and unmatched products, resulting in an incomplete analysis.

We will:
- Report final dimensions.
- Count and characterize unmatched records on each side.
- Verify key cardinality to confirm that no row duplication occurred.
- Compute `Revenue = Quantity * UnitPrice`. Transactions matching missing products will have NA revenue and are filtered out for sales analysis.

In [ ]:
# Perform left joins
joined_step1 <- transactions_cleaned %>% left_join(products_cleaned, by = "StockCode")
joined_final <- joined_step1 %>% left_join(customers_cleaned, by = "CustomerID")

cat("Final integrated dataset dimensions:", nrow(joined_final), "rows x", ncol(joined_final), "columns\n")

# Characterise unmatched records
unmatched_products <- joined_final %>% filter(is.na(UnitPrice))
cat("Unmatched transactions due to missing products:", nrow(unmatched_products), "\n")
cat("Unique unmatched StockCodes:", length(unique(unmatched_products$StockCode)), "\n")

unmatched_customers <- joined_final %>% filter(is.na(Country) & !is.na(CustomerID))
cat("Unmatched transactions with non-NA CustomerID (missing from customers directory):", nrow(unmatched_customers), "\n")

na_customer_transactions <- joined_final %>% filter(is.na(CustomerID))
cat("Transactions with missing CustomerID:", nrow(na_customer_transactions), "\n")

# Cardinality Verification
cat("Verification check: Cleaned transactions count =", nrow(transactions_cleaned), "and Joined count =", nrow(joined_final), "\n")
if (nrow(transactions_cleaned) == nrow(joined_final)) {
  cat("Verification SUCCESS: No row duplication occurred during the join.\n")
} else {
  warning("Verification FAILURE: Row duplication detected!")
}

# Calculate Revenue and filter for analysis
joined_final <- joined_final %>% mutate(Revenue = Quantity * UnitPrice)
analysis_data <- joined_final %>% filter(!is.na(Revenue))
cat("Transactions with valid revenue for analysis:", nrow(analysis_data), "\n")

## Task 3: Sales and Customer Analysis
We compute:
1. Total sales revenue.
2. Top 5 products by revenue.
3. Top 5 countries by revenue.
4. Top 5 customers by total purchase value.

We segment customers into `Low Value`, `Medium Value`, `High Value`, and `Premium` based on the spend distribution's 50th, 75th, and 95th percentiles. We analyze the market, explaining how the Netherlands represents a high-performing market, and Saudi Arabia represents an underperforming market, taking into account the volume dominance of the United Kingdom.

In [ ]:
# 1. Total sales revenue
total_revenue <- sum(analysis_data$Revenue)
cat("Total Sales Revenue: $", format(total_revenue, big.mark = ","), "\n")

# 2. Top 5 products by revenue
top_products <- analysis_data %>% 
  group_by(StockCode, Description) %>% 
  summarise(Revenue = sum(Revenue), .groups = "drop") %>% 
  arrange(desc(Revenue)) %>% 
  slice(1:5)
cat("\nTop 5 Products by Revenue:\n")
print(top_products)

# 3. Top 5 countries by revenue
top_countries <- analysis_data %>% 
  group_by(Country) %>% 
  summarise(Revenue = sum(Revenue), .groups = "drop") %>% 
  arrange(desc(Revenue)) %>% 
  slice(1:5)
cat("\nTop 5 Countries by Revenue:\n")
print(top_countries)

# 4. Top 5 customers by revenue (non-NA)
top_customers <- analysis_data %>% 
  filter(!is.na(CustomerID)) %>% 
  group_by(CustomerID) %>% 
  summarise(Revenue = sum(Revenue), .groups = "drop") %>% 
  arrange(desc(Revenue)) %>% 
  slice(1:5)
cat("\nTop 5 Customers by Revenue:\n")
print(top_customers)

# Customer Segmentation
customer_spend <- analysis_data %>% 
  filter(!is.na(CustomerID)) %>% 
  group_by(CustomerID) %>% 
  summarise(TotalSpend = sum(Revenue), .groups = "drop")

thresholds <- quantile(customer_spend$TotalSpend, probs = c(0.50, 0.75, 0.95))
cat("\nSpend Percentiles:\n")
print(thresholds)

customer_spend <- customer_spend %>% 
  mutate(Segment = case_when(
    TotalSpend <= thresholds[1] ~ "Low Value",
    TotalSpend > thresholds[1] & TotalSpend <= thresholds[2] ~ "Medium Value",
    TotalSpend > thresholds[2] & TotalSpend <= thresholds[3] ~ "High Value",
    TotalSpend > thresholds[3] ~ "Premium"
  ))

segment_summary <- customer_spend %>% 
  group_by(Segment) %>% 
  summarise(Customer_Count = n(), Total_Spend = sum(TotalSpend), Average_Spend = mean(TotalSpend), .groups = "drop") %>% 
  mutate(Segment = factor(Segment, levels = c("Low Value", "Medium Value", "High Value", "Premium"))) %>% 
  arrange(Segment)
cat("\nCustomer Segment Summary:\n")
print(segment_summary)

# Country analysis for markets
country_metrics <- analysis_data %>% 
  group_by(Country) %>% 
  summarise(
    TotalRevenue = sum(Revenue),
    TotalTransactions = n_distinct(InvoiceNo),
    TotalCustomers = n_distinct(CustomerID, na.rm = TRUE),
    RevenuePerCustomer = if_else(TotalCustomers > 0, TotalRevenue / TotalCustomers, NA_real_),
    RevenuePerTransaction = TotalRevenue / TotalTransactions,
    .groups = "drop"
  ) %>% 
  arrange(desc(TotalRevenue))
cat("\nDetailed Top 10 Country Metrics:\n")
print(head(country_metrics, 10))

### Plotting Sales Insights
We generate two plots and save them to `output/` using `ggsave()`. Running these code blocks will also display them inline in the notebook.

In [ ]:
# Plot 1: Top 5 Products by Revenue
p1 <- ggplot(top_products, aes(x = reorder(Description, Revenue), y = Revenue, fill = Revenue)) +
  geom_bar(stat = "identity", width = 0.6) +
  coord_flip() +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_fill_gradient(low = "#5c7cfa", high = "#1a365d") +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14, color = "#1a202c"),
    plot.subtitle = element_text(size = 10, color = "#4a5568", margin = margin(b = 10)),
    axis.title = element_text(size = 11, face = "bold", color = "#2d3748"),
    axis.text = element_text(size = 9, color = "#4a5568"),
    panel.grid.major.y = element_blank(),
    legend.position = "none"
  ) +
  labs(
    title = "Top 5 Products by Sales Revenue",
    subtitle = "Online Retail Data Analysis",
    x = "Product Description",
    y = "Total Revenue"
  )
print(p1)
ggsave("output/plot1_top_products.png", plot = p1, width = 8, height = 4.5, dpi = 300)

# Plot 2: Customer Value Segment Distribution
p2 <- ggplot(segment_summary, aes(x = Segment, y = Customer_Count, fill = Segment)) +
  geom_bar(stat = "identity", width = 0.5) +
  geom_text(aes(label = format(Customer_Count, big.mark = ",")), vjust = -0.5, size = 3.5, fontface = "bold") +
  scale_fill_manual(values = c("Low Value" = "#cbd5e0", "Medium Value" = "#a3b18a", "High Value" = "#457b9d", "Premium" = "#1d3557")) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14, color = "#1a202c"),
    plot.subtitle = element_text(size = 10, color = "#4a5568", margin = margin(b = 10)),
    axis.title = element_text(size = 11, face = "bold", color = "#2d3748"),
    axis.text = element_text(size = 10, color = "#4a5568"),
    panel.grid.major.x = element_blank(),
    legend.position = "none"
  ) +
  labs(
    title = "Distribution of Customers Across Value Segments",
    subtitle = "Segmentation thresholds derived from total spend distribution percentiles",
    x = "Customer Segment",
    y = "Number of Customers"
  )
print(p2)
ggsave("output/plot2_customer_segments.png", plot = p2, width = 8, height = 4.5, dpi = 300)

## Task 4: SQLite Database Storage and Cross-Verification
We create a SQLite database at `output/retail_sales.db` and write the integrated dataset to the `retail_sales` table. We then execute two SQL queries to find the top 5 customers and top 5 countries by revenue, validating these SQL results against the dplyr results computed in Task 3. Finally, we report the size of the database file.

In [ ]:
db_path <- "output/retail_sales.db"
if (file.exists(db_path)) {
  file.remove(db_path)
}

con <- dbConnect(SQLite(), db_path)

# Write table to SQLite. Date needs to be cast to character for compatibility.
sqlite_data <- joined_final %>% mutate(InvoiceDate = as.character(InvoiceDate))
dbWriteTable(con, "retail_sales", sqlite_data, overwrite = TRUE)

# Run Query 1: Top 5 customers by revenue
sql_customers <- dbGetQuery(con, "
  SELECT CustomerID, SUM(Quantity * UnitPrice) as Revenue
  FROM retail_sales
  WHERE CustomerID IS NOT NULL AND UnitPrice IS NOT NULL AND Quantity IS NOT NULL
  GROUP BY CustomerID
  ORDER BY Revenue DESC
  LIMIT 5
")
cat("SQL Query 1: Top 5 Customers by Revenue:\n")
print(sql_customers)

# Run Query 2: Total revenue by country
sql_countries <- dbGetQuery(con, "
  SELECT Country, SUM(Quantity * UnitPrice) as Revenue
  FROM retail_sales
  WHERE UnitPrice IS NOT NULL AND Quantity IS NOT NULL
  GROUP BY Country
  ORDER BY Revenue DESC
  LIMIT 5
")
cat("\nSQL Query 2: Total Revenue by Country (Top 5):\n")
print(sql_countries)

dbDisconnect(con)

# Parity validation
dplyr_top_cust <- top_customers %>% mutate(CustomerID = as.numeric(CustomerID))
sql_top_cust <- sql_customers %>% mutate(CustomerID = as.numeric(CustomerID))
cust_match <- all.equal(dplyr_top_cust$CustomerID, sql_top_cust$CustomerID) &&
              all.equal(round(dplyr_top_cust$Revenue, 2), round(sql_top_cust$Revenue, 2))

dplyr_top_country <- top_countries
sql_top_country <- sql_countries
country_match <- all.equal(dplyr_top_country$Country, sql_top_country$Country) &&
                 all.equal(round(dplyr_top_country$Revenue, 2), round(sql_top_country$Revenue, 2))

cat("\nParity Cross-Check Results:\n")
cat("- Top 5 Customers matches exactly:", cust_match, "\n")
cat("- Top 5 Countries matches exactly:", country_match, "\n")

# Database file size reporting
db_size <- file.info(db_path)$size
cat("\nDatabase file size:", round(db_size / (1024 * 1024), 2), "MB\n")
if (db_size > 50 * 1024 * 1024) {
  cat("WARNING: The SQLite database exceeds 50 MB.\n")
} else {
  cat("Database file size is under 50 MB. Safe for commit.\n")
}